# DeepWeb Dataset Builder: High-Integrity Pretraining Curation

This notebook ingests raw or educational web data (e.g., `HuggingFaceFW/fineweb-edu`), runs the **Discrete Evidence Engine (DEE)** compiled in Rust for mathematical and epistemic verification, and outputs the **DeepWeb** pretraining dataset partitioned into the **SmolLM2 4-stage developmental curriculum**:
- **Stage 1 (Fundamental)**: Foundational narrative & simple encyclopedic concepts
- **Stage 2 (Intermediate)**: High school & explanatory STEM/humanities
- **Stage 3 (Advanced)**: Undergraduate textbooks, legal codes & technical specifications
- **Stage 4 (Research)**: Peer-reviewed research, formal mathematical proofs & patents

In [ ]:
# 1. Install prerequisites
!pip install -q datasets pyarrow pandas

In [ ]:
# 2. Locate and stage DEE standalone binary into working directory
import os, sys, shutil, glob, subprocess

print("Available input directories:", glob.glob("/kaggle/input/*"))
candidates = glob.glob("/kaggle/input/**/dee", recursive=True) + glob.glob("../input/**/dee", recursive=True) + glob.glob("./dee")
print("Found binary candidates:", candidates)

dee_bin = "/kaggle/working/dee"
if candidates:
    src_bin = candidates[0]
    if os.path.abspath(src_bin) != os.path.abspath(dee_bin):
        shutil.copy2(src_bin, dee_bin)
    os.chmod(dee_bin, 0o755)
    print(f"Successfully staged executable DEE binary: {dee_bin}")
else:
    print("Listing all files in inputs:", glob.glob("/kaggle/input/**/*", recursive=True))
    raise FileNotFoundError("DEE binary not found in /kaggle/input")

!{dee_bin} --help


In [ ]:
# 3. Locate builder script and execute DeepWeb Ingestion Pipeline
import os, glob

builder_candidates = glob.glob("/kaggle/input/**/deepweb_builder.py", recursive=True) + glob.glob("../input/**/deepweb_builder.py", recursive=True) + glob.glob("./deepweb_builder.py")
if builder_candidates:
    builder_script = builder_candidates[0]
else:
    builder_script = "deepweb_builder.py"
print(f"Using builder script: {builder_script}")

!python3 {builder_script} --max-documents 25000 --chunk-size 5000 --output-dir /kaggle/working/deepweb --dee-binary {dee_bin}


In [ ]:
# 4. Inspect Partitioned DeepWeb Dataset
import pandas as pd
from pathlib import Path

staged_dir = Path("/kaggle/working/deepweb/staged_curriculum")
for f in sorted(staged_dir.glob("*.csv")):
    df = pd.read_csv(f)
    parquet_path = f.with_suffix(".parquet")
    df.to_parquet(parquet_path, compression="snappy")
    print(f"{f.name:<30}: {len(df):>6} docs | Parquet size: {parquet_path.stat().st_size / (1024*1024):.2f} MB")

print("\nDeepWeb dataset successfully generated and ready for pretraining!")